# 05 - Estrategia: Breakout con Canales de Donchian

**Capítulo**: 05 - Breakout

**Objetivo**: Implementar ruptura de canales Donchian con trailing stop ATR.

---

In [ ]:
import sys
sys.path.insert(0, '../..')

import pandas as pd
import matplotlib.pyplot as plt
from curso.lib.data import download_historical
from curso.lib.indicators import donchian, atr

plt.style.use('seaborn-v0_8-whitegrid')
print('Setup completado ✓')

## 1. Datos

In [ ]:
TICKER = 'GLD'  # Oro ETF - buen candidato para breakout
df = download_historical(TICKER)
print(f'{TICKER}: {len(df)} registros')

## 2. Canales de Donchian

In [ ]:
ENTRY_PERIOD = 20
EXIT_PERIOD = 10
ATR_PERIOD = 14

dc = donchian(df, length=ENTRY_PERIOD)
df['DC_upper'] = dc['upper']
df['DC_lower'] = dc['lower']
df['DC_middle'] = dc['middle']
df['ATR'] = atr(df, length=ATR_PERIOD)

# Canal de salida (más corto)
dc_exit = donchian(df, length=EXIT_PERIOD)
df['DC_exit_lower'] = dc_exit['lower']

# Señales de breakout
df['breakout_up'] = df['Close'] > df['DC_upper'].shift(1)
df['breakdown'] = df['Close'] < df['DC_exit_lower'].shift(1)

print(f'Breakouts alcistas: {df["breakout_up"].sum()}')
print(f'Breakdowns: {df["breakdown"].sum()}')
print(f'ATR promedio: ${df["ATR"].mean():.2f}')

## 3. Visualización

In [ ]:
data = df.tail(300).copy()

fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(data.index, data['Close'], linewidth=1.2, color='#333', label='Precio')
ax.plot(data.index, data['DC_upper'], color='#F44336', linewidth=0.8, label=f'Donchian({ENTRY_PERIOD}) Sup')
ax.plot(data.index, data['DC_lower'], color='#4CAF50', linewidth=0.8, label=f'Donchian({ENTRY_PERIOD}) Inf')
ax.fill_between(data.index, data['DC_lower'], data['DC_upper'], alpha=0.05, color='blue')

breakouts = data[data['breakout_up']]
ax.scatter(breakouts.index, breakouts['Close'], marker='^', color='green', s=100, zorder=5, label='Breakout')

ax.set_title(f'{TICKER} - Donchian Channel ({ENTRY_PERIOD})', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Cálculo de Position Sizing

In [ ]:
CAPITAL = 10_000
RISK_PER_TRADE = 0.02  # 2%
ATR_MULTIPLIER = 3.0

# Ejemplo de sizing
latest_atr = df['ATR'].iloc[-1]
risk_amount = CAPITAL * RISK_PER_TRADE
stop_distance = ATR_MULTIPLIER * latest_atr
position_size = risk_amount / stop_distance
position_value = position_size * df['Close'].iloc[-1]

print(f'Capital: ${CAPITAL:,.0f}')
print(f'Riesgo por trade: ${risk_amount:.0f} ({RISK_PER_TRADE*100}%)')
print(f'ATR actual: ${latest_atr:.2f}')
print(f'Stop distance: ${stop_distance:.2f} ({ATR_MULTIPLIER}×ATR)')
print(f'Tamaño posición: {position_size:.1f} acciones')
print(f'Valor posición: ${position_value:,.0f} ({position_value/CAPITAL*100:.1f}% del capital)')

## Resumen

✅ Calculados canales de Donchian
✅ Detectadas rupturas (breakouts)
✅ Implementado position sizing basado en ATR
✅ Visualizado canal y señales

**Siguiente**: `05_backtesting.ipynb`